# 02_事件流

**来源:** [LangChain Docs — Event Streaming](https://docs.langchain.com/oss/python/deepagents/event-streaming)

本 Notebook 演示 Deep Agents 的事件流（Event Streaming）API。事件流是 Deep Agents v0.6 引入的类型化投影 API，提供独立的迭代器用于子 Agent、消息、工具调用和值，无需在 `stream_mode` 块上进行分支判断。

## 1. 环境准备与导入

In [ ]:
# pip install deepagents langchain langgraph

from deepagents import create_deep_agent

## 2. 流式获取子 Agent

Deep Agents 在 LangGraph 流式之上增加了子 Agent 投影。使用 `stream.subagents` 获取每个委托任务调用的独立流句柄。

In [ ]:
# 创建深度 Agent
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt="你是一个有用的助手",
    subagents=[
        {"name": "writer", "description": "创意写作", "system_prompt": "你是一个诗人。"}
    ],
)

# 使用 stream_events API 获取子 Agent 流
stream = agent.stream_events({
    "messages": [{"role": "user", "content": "写一首关于大海的俳句"}],
}, version="v3")

# 遍历子 Agent 流
for subagent in stream.subagents:
    print(f"名称: {subagent.name}, 路径: {subagent.path}, 状态: {subagent.status}")

    # 获取子 Agent 的消息
    for message in subagent.messages:
        print(f"  消息: {message.text}")

## 3. 子 Agent 流字段

每个子 Agent 流暴露的投影与父运行相同，包括消息、工具调用、嵌套子 Agent 和最终输出。

| 字段 | 描述 |
|------|------|
| `name` | 子 Agent 名称 |
| `messages` | 子 Agent 发出的消息 |
| `subagents` | 嵌套的子 Agent 调用 |
| `output` | 子 Agent 最终状态或委托任务的完成信号 |
| `path` | 子 Agent 流的命名空间路径 |
| `status` | 生命周期状态（started, completed, failed, interrupted） |
| `tool_calls` | 子 Agent 范围内的工具调用 |

## 4. 追踪子 Agent 生命周期

使用 `stream.subagents` 仅需展示哪些子 Agent 启动和完成，无需订阅消息或值流。

In [ ]:
# 追踪子 Agent 生命周期
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "写一首关于大海的俳句"}]},
    version="v3"
)

running = 0
completed = 0
failed = 0

for subagent in stream.subagents:
    running += 1
    print(f"{subagent.name}: 已启动")

    try:
        _ = subagent.output
        running -= 1
        completed += 1
        print(f"{subagent.name}: 已完成")
    except Exception:
        running -= 1
        failed += 1
        print(f"{subagent.name}: 失败")

## 5. 流式获取消息

Deep Agents 可以从协调 Agent 和委托的子 Agent 发出消息。使用顶层 `stream.messages` 获取主消息，使用 `subagent.messages` 获取每个子 Agent 的消息。

In [ ]:
# 流式获取消息
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "写一首关于大海的俳句"}]},
    version="v3"
)

# 获取协调 Agent 的消息
for message in stream.messages:
    print("[协调]", message.text)

# 获取每个子 Agent 的消息
for subagent in stream.subagents:
    for message in subagent.messages:
        print(f"[{subagent.name}]", message.text)

## 6. 流式获取工具调用

Deep Agents 在 Agent 树的每个层级暴露工具调用。使用顶层 `stream.tool_calls` 获取协调工具，使用 `subagent.tool_calls` 获取委托工作。

In [ ]:
# 流式获取工具调用
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "写一首关于大海的俳句"}]},
    version="v3"
)

# 协调 Agent 的工具调用
for call in stream.tool_calls:
    print("[协调工具]", call.tool_name, call.input)
    print("  完成:", call.completed, "错误:", call.error)

# 每个子 Agent 的工具调用
for subagent in stream.subagents:
    for call in subagent.tool_calls:
        print(f"[{subagent.name} 工具]", call.tool_name, call.input)
        for delta in call.output_deltas:
            print(delta, end="", flush=True)

        if call.completed and call.error is None:
            print("输出:", call.output)
        elif call.error is not None:
            print("错误:", call.error)

## 7. 流式获取嵌套工作

可以递归进入子 Agent 流观察嵌套的子 Agent、消息和工具调用。

In [ ]:
# 遍历嵌套的子 Agent 工作
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "写一首关于大海的俳句"}]},
    version="v3"
)

for subagent in stream.subagents:
    print(f"子 Agent {subagent.name}: {subagent.status}")

    # 工具调用
    for tool_call in subagent.tool_calls:
        print(f"  {tool_call.tool_name}({tool_call.input})")
        for delta in tool_call.output_deltas:
            print(f"    {delta}", end="", flush=True)

    # 嵌套子 Agent
    for nested in subagent.subagents:
        print(f"  嵌套子 Agent {nested.name}: {nested.status}")

## 8. 并发消费

协调 Agent 和子 Agent 的输出通常是交错的。需要实时 UI 更新时应并发消费投影。


### 同步代码：使用 stream.interleave

对于同步代码，使用 `stream.interleave(...)` 交错消费多个投影。

In [ ]:
# 使用 interleave 交错消费
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "写一首关于大海的俳句"}]},
    version="v3"
)

for name, item in stream.interleave("messages", "subagents"):
    if name == "messages":
        print("[协调]", item.text)
    else:
        for message in item.messages:
            print(f"[{item.name}]", message.text)

### 原始协议事件

当需要在协调 Agent 和所有子 Agent 之间保持精确到达顺序时，直接迭代原始协议事件并使用 namespace 标识来源。

In [ ]:
# 使用原始协议事件遍历
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "写一首关于大海的俳句"}]},
    version="v3"
)

for event in stream:
    if event.get("method") != "messages":
        continue

    payload = event["params"]["data"][0]
    if not isinstance(payload, dict):
        continue
    if payload.get("event") != "content-block-delta":
        continue

    block = payload.get("delta") or {}
    if block.get("type") == "text-delta":
        # 根据 namespace 判断来源
        source = "子 Agent" if event["params"]["namespace"] else "协调"
        print(f"[{source}] {block['text']}")

## 9. 子 Agent 与子图的区别

- `stream.subgraphs` 展示图执行结构
- `stream.subagents` 展示产品级的 Deep Agents 任务委托

对于面向用户的 UI，应使用 `stream.subagents`，因为它隐藏了内部图节点，直接暴露子 Agent 概念。

---

**参考链接**
- [Deep Agents — Event Streaming](https://docs.langchain.com/oss/python/deepagents/event-streaming)
- [LangChain Event Streaming](https://docs.langchain.com/oss/python/deepagents/event-streaming)
- [子 Agent 前端流式](https://docs.langchain.com/oss/python/deepagents/frontend-streaming)
- [LangGraph Event Streaming](https://langchain-ai.github.io/langgraph/)